In [1]:
# IBM Watson Machine Learning Deployment Pipeline
# Credit Card Approval Prediction Model

# Step 1: Install required IBM Watson libraries
# Run this in terminal: pip install ibm-watson-machine-learning

# Import required libraries
import joblib
import pandas as pd
import numpy as np
import json

In [2]:
# Step 2: Load our trained XGBoost model
model = joblib.load("credit_approval_model.pkl")
model_columns = joblib.load("model_columns.pkl")

print("Model loaded successfully")
print(f"Model type: {type(model)}")
print(f"Number of features: {len(model_columns)}")
print(f"Features: {model_columns[:5]}...")

Model loaded successfully
Model type: <class 'xgboost.sklearn.XGBClassifier'>
Number of features: 47
Features: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL']...


In [3]:
# Step 3: IBM Watson ML Configuration
# These credentials are obtained from IBM Cloud Dashboard
# Watson Machine Learning Service → Service Credentials → New Credential

WML_CREDENTIALS = {
    "apikey": "YOUR_IBM_CLOUD_API_KEY",        # From IBM Cloud → Manage → API Keys
    "url": "https://us-south.ml.cloud.ibm.com"  # Dallas region endpoint
}

SPACE_ID = "YOUR_DEPLOYMENT_SPACE_ID"  # From Watson Studio → Deployment Spaces

print("Watson ML Credentials configured")
print(f"WML URL: {WML_CREDENTIALS['url']}")

Watson ML Credentials configured
WML URL: https://us-south.ml.cloud.ibm.com


In [4]:
# Step 4: Connect to IBM Watson Machine Learning
# from ibm_watson_machine_learning import APIClient

# Initialize Watson ML client
# client = APIClient(WML_CREDENTIALS)
# client.set.default_space(SPACE_ID)
# print("Connected to Watson ML successfully")
# print(client.version)

# NOTE: Uncomment above code after:
# 1. Installing: pip install ibm-watson-machine-learning
# 2. Adding real API credentials above
# 3. Creating a deployment space in Watson Studio

print("Watson ML Client ready to initialize")
print("Status: Awaiting credentials configuration")

Watson ML Client ready to initialize
Status: Awaiting credentials configuration


In [5]:
# Step 5: Save model in Watson ML compatible format
# Watson ML supports scikit-learn pipeline format

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a simple pipeline with our model
# (Watson ML requires pipeline format for deployment)
wml_pipeline = Pipeline([
    ('model', model)
])

# Save pipeline for Watson upload
joblib.dump(wml_pipeline, "watson_model_pipeline.pkl")
print("Model pipeline saved for Watson ML upload")

# Model metadata for Watson
model_metadata = {
    "name": "Credit Card Approval Prediction",
    "type": "binary_classification",
    "algorithm": "XGBoost",
    "features": model_columns.tolist() if hasattr(model_columns, 'tolist') else list(model_columns),
    "target": "TARGET",
    "classes": {0: "APPROVED (Low Risk)", 1: "REJECTED (High Risk)"},
    "accuracy": 0.7885,
    "recall_risky_class": 0.6200,
    "training_samples": 29165,
    "test_samples": 7292
}

with open("watson_model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=4)

print("Model metadata saved:")
print(json.dumps(model_metadata, indent=4))

Model pipeline saved for Watson ML upload
Model metadata saved:
{
    "name": "Credit Card Approval Prediction",
    "type": "binary_classification",
    "algorithm": "XGBoost",
    "features": [
        "CODE_GENDER",
        "FLAG_OWN_CAR",
        "FLAG_OWN_REALTY",
        "CNT_CHILDREN",
        "AMT_INCOME_TOTAL",
        "FLAG_WORK_PHONE",
        "FLAG_PHONE",
        "FLAG_EMAIL",
        "CNT_FAM_MEMBERS",
        "AGE_YEARS",
        "IS_UNEMPLOYED",
        "YEARS_EMPLOYED",
        "NAME_INCOME_TYPE_Pensioner",
        "NAME_INCOME_TYPE_State servant",
        "NAME_INCOME_TYPE_Student",
        "NAME_INCOME_TYPE_Working",
        "NAME_EDUCATION_TYPE_Higher education",
        "NAME_EDUCATION_TYPE_Incomplete higher",
        "NAME_EDUCATION_TYPE_Lower secondary",
        "NAME_EDUCATION_TYPE_Secondary / secondary special",
        "NAME_FAMILY_STATUS_Married",
        "NAME_FAMILY_STATUS_Separated",
        "NAME_FAMILY_STATUS_Single / not married",
        "NAME_FAMILY_S

In [6]:
# Step 6: Watson ML Deployment Code
# (Uncomment after configuring real credentials)

"""
DEPLOYMENT STEPS:
=================

1. Store the model in Watson ML repository:
   model_artifact = client.repository.store_model(
       model=wml_pipeline,
       meta_props={
           client.repository.ModelMetaNames.NAME: "credit_approval_model",
           client.repository.ModelMetaNames.TYPE: "scikit-learn_1.1",
           client.repository.ModelMetaNames.SOFTWARE_SPEC_UID:
               client.software_specifications.get_id_by_name("runtime-22.2-py3.10")
       }
   )
   model_uid = client.repository.get_model_id(model_artifact)
   print(f"Model stored with UID: {model_uid}")

2. Deploy the model as online endpoint:
   deployment = client.deployments.create(
       artifact_uid=model_uid,
       meta_props={
           client.deployments.ConfigurationMetaNames.NAME: "credit_approval_deployment",
           client.deployments.ConfigurationMetaNames.ONLINE: {}
       }
   )
   deployment_uid = client.deployments.get_uid(deployment)
   print(f"Model deployed with UID: {deployment_uid}")

3. Get scoring endpoint URL:
   scoring_url = client.deployments.get_scoring_href(deployment)
   print(f"Scoring URL: {scoring_url}")

4. Test the deployment with sample data:
   scoring_payload = {
       "input_data": [{
           "fields": model_columns,
           "values": [X_test.iloc[0].tolist()]
       }]
   }
   predictions = client.deployments.score(deployment_uid, scoring_payload)
   print(f"Sample prediction: {predictions}")
"""

print("Watson ML Deployment pipeline code ready")
print("To activate: Add IBM Cloud credentials and uncomment the code above")

Watson ML Deployment pipeline code ready
To activate: Add IBM Cloud credentials and uncomment the code above


In [7]:
# Step 7: Flask App Integration with Watson ML Endpoint
# This shows how app.py would use Watson cloud instead of local model

watson_flask_code = """
# Replace local model loading in app.py with Watson ML scoring:

from ibm_watson_machine_learning import APIClient

WML_CREDENTIALS = {
    "apikey": "YOUR_API_KEY",
    "url": "https://us-south.ml.cloud.ibm.com"
}

client = APIClient(WML_CREDENTIALS)
DEPLOYMENT_UID = "YOUR_DEPLOYMENT_UID"

def predict_with_watson(input_df):
    scoring_payload = {
        "input_data": [{
            "fields": input_df.columns.tolist(),
            "values": input_df.values.tolist()
        }]
    }
    result = client.deployments.score(DEPLOYMENT_UID, scoring_payload)
    prediction = result['predictions'][0]['values'][0][0]
    probability = result['predictions'][0]['values'][0][1][1]
    return prediction, probability
"""

print("Watson ML Flask Integration Code:")
print(watson_flask_code)
print("="*50)
print("DEPLOYMENT PIPELINE SUMMARY")
print("="*50)
print("1. Model trained locally using XGBoost")
print("2. Model saved as .pkl file")
print("3. Model uploaded to IBM Watson ML repository")
print("4. Online deployment created with scoring endpoint")
print("5. Flask app updated to call Watson scoring URL")
print("6. Real-time predictions served from IBM Cloud")

Watson ML Flask Integration Code:

# Replace local model loading in app.py with Watson ML scoring:

from ibm_watson_machine_learning import APIClient

WML_CREDENTIALS = {
    "apikey": "YOUR_API_KEY",
    "url": "https://us-south.ml.cloud.ibm.com"
}

client = APIClient(WML_CREDENTIALS)
DEPLOYMENT_UID = "YOUR_DEPLOYMENT_UID"

def predict_with_watson(input_df):
    scoring_payload = {
        "input_data": [{
            "fields": input_df.columns.tolist(),
            "values": input_df.values.tolist()
        }]
    }
    result = client.deployments.score(DEPLOYMENT_UID, scoring_payload)
    prediction = result['predictions'][0]['values'][0][0]
    probability = result['predictions'][0]['values'][0][1][1]
    return prediction, probability

DEPLOYMENT PIPELINE SUMMARY
1. Model trained locally using XGBoost
2. Model saved as .pkl file
3. Model uploaded to IBM Watson ML repository
4. Online deployment created with scoring endpoint
5. Flask app updated to call Watson scoring URL
6. Real-t